# Official HEC-HMS Guide Mirror: Basin Methods, Loss, Transform, Baseflow, and Routing

Official guides:

- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/parameter-estimation
- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-loss-methods-in-hec-hms
- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-transform-methods-in-hec-hms
- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-baseflow-methods-in-hec-hms
- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-reach-routing-methods-within-hec-hms

This notebook uses one extracted HMS sample project to inspect and export the basin parameters that those guide categories manipulate in the GUI.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import HmsBasin, HmsUtils

project, project_path = init_sample_project("castro", "25_basin_methods")
basin_path = Path(project.basin_df.iloc[0]["full_path"])

loss = HmsBasin.get_all_loss_parameters(basin_path)
transform = HmsBasin.get_all_transform_parameters(basin_path)
baseflow = HmsBasin.get_all_baseflow_parameters(basin_path)
routing = HmsBasin.get_all_routing_parameters(basin_path)

method_summary = pd.DataFrame([
    {"category": "loss", "elements": len(loss), "methods": ", ".join(sorted(loss["loss_method"].dropna().astype(str).unique()))},
    {"category": "transform", "elements": len(transform), "methods": ", ".join(sorted(transform["transform_method"].dropna().astype(str).unique()))},
    {"category": "baseflow", "elements": len(baseflow), "methods": ", ".join(sorted(baseflow["baseflow_method"].dropna().astype(str).unique()))},
    {"category": "routing", "elements": len(routing), "methods": ", ".join(sorted(routing["route_method"].dropna().astype(str).unique()))},
])
assert method_summary["elements"].sum() > 0
method_summary

,category,elements,methods
0,loss,4,Initial+Constant
1,transform,4,Snyder
2,baseflow,4,Recession
3,routing,2,"Modified Puls, Muskingum"


In [3]:
curve_number_examples = pd.DataFrame({"curve_number": [60, 75, 85, 95]})
curve_number_examples["initial_abstraction_in"] = curve_number_examples["curve_number"].apply(HmsUtils.calculate_ia_from_cn)
curve_number_examples["roundtrip_curve_number"] = curve_number_examples["initial_abstraction_in"].apply(HmsUtils.calculate_cn_from_ia)
curve_number_examples["roundtrip_error"] = (curve_number_examples["curve_number"] - curve_number_examples["roundtrip_curve_number"]).abs()
assert curve_number_examples["roundtrip_error"].max() < 1e-9
curve_number_examples

,curve_number,initial_abstraction_in,roundtrip_curve_number,roundtrip_error
0,60,1.333333,60.0,7.105427e-15
1,75,0.666667,75.0,0.000000e+00
2,85,0.352941,85.0,0.000000e+00
3,95,0.105263,95.0,0.000000e+00


In [4]:
export_csv = WORK_ROOT / "25_basin_methods" / "parameter_exports" / "castro_parameters.csv"
export_csv.parent.mkdir(parents=True, exist_ok=True)
combined_csv = HmsBasin.export_parameters_csv(basin_path, export_csv)
exported_files = sorted(export_csv.parent.glob("castro_parameters*.csv"))
export_summary = pd.DataFrame([
    {"file": path.name, "exists": path.exists(), "size_kb": round(path.stat().st_size / 1024, 1)}
    for path in exported_files
])
assert combined_csv.exists()
assert len(export_summary) >= 4
export_summary

,file,exists,size_kb
0,castro_parameters.csv,True,1.2
1,castro_parameters_baseflow.csv,True,0.8
2,castro_parameters_loss.csv,True,0.8
3,castro_parameters_routing.csv,True,0.7
4,castro_parameters_transform.csv,True,0.5


In [5]:
preview_columns = {
    "loss": ["name", "loss_method", "percent_impervious", "curve_number", "initial_abstraction"],
    "transform": ["name", "transform_method", "time_of_concentration", "storage_coefficient", "lag"],
    "baseflow": ["name", "baseflow_method", "recession_factor", "initial_discharge"],
    "routing": ["name", "route_method", "muskingum_k", "muskingum_x", "lag"],
}
previews = []
for label, frame in [("loss", loss), ("transform", transform), ("baseflow", baseflow), ("routing", routing)]:
    cols = [col for col in preview_columns[label] if col in frame.columns]
    sample = frame[cols].head(3).copy()
    sample.insert(0, "category", label)
    previews.append(sample)
pd.concat(previews, ignore_index=True, sort=False)

,category,name,loss_method,percent_impervious,curve_number,initial_abstraction,transform_method,time_of_concentration,storage_coefficient,lag,baseflow_method,recession_factor,initial_discharge,route_method,muskingum_k,muskingum_x
0,loss,Subbasin-3,Initial+Constant,10.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,loss,Subbasin-4,Initial+Constant,15.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,loss,Subbasin-1,Initial+Constant,2.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,transform,Subbasin-3,NaN,NaN,NaN,NaN,Snyder,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
4,transform,Subbasin-4,NaN,NaN,NaN,NaN,Snyder,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
5,transform,Subbasin-1,NaN,NaN,NaN,NaN,Snyder,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
6,baseflow,Subbasin-3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Recession,0.79,<NA>,NaN,NaN,NaN
7,baseflow,Subbasin-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Recession,0.79,<NA>,NaN,NaN,NaN
8,baseflow,Subbasin-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Recession,0.79,<NA>,NaN,NaN,NaN
9,routing,Reach-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Modified Puls,NaN,NaN


## Coverage Notes

This starter mirror validates the generic batch parameter surface for loss, transform, baseflow, and routing methods. Method-specific calculators and soil/land-use raster processing for official parameter-estimation tutorials are tracked in CLB-289.